# ANRF AISEHack 2.0 - Polymer Property Prediction
**Approach:** RDKit descriptors + Morgan fingerprints + MACCS keys → LightGBM/XGBoost ensemble with Optuna tuning

**Targets:** Tg (glass transition temperature, °C) and Egc (chain band gap, eV) — predicted separately.

In [34]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# RDKit
from rdkit import Chem
from rdkit.Chem import (
    Descriptors,
    AllChem,
    MACCSkeys
)

# sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.ensemble import GradientBoostingRegressor

# Boosting models
import lightgbm as lgb
import xgboost as xgb

# Reproducibility
SEED = 42
np.random.seed(SEED)

print("Imports done.")
print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print(f"XGBoost version: {xgb.__version__}")

Imports done.
Numpy version: 2.4.6
Pandas version: 2.3.3
LightGBM version: 4.6.0
XGBoost version: 3.2.0


## 1. Load Data

In [35]:
train = pd.read_csv('/kaggle/input/datasets/josephjonathanfdes/jjf-aise2/train.csv')
test = pd.read_csv('/kaggle/input/datasets/josephjonathanfdes/jjf-aise2/test.csv')

print("=" * 50)
print("Dataset Shapes")
print("=" * 50)

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")

print("\n" + "=" * 50)
print("Target Type Distribution")
print("=" * 50)

print("\nTrain:")
print(train['target_type'].value_counts())

print("\nTest:")
print(test['target_type'].value_counts())

print("\n" + "=" * 50)
print("Missing Values")
print("=" * 50)

print("\nTrain:")
print(train.isnull().sum()[train.isnull().sum() > 0])

print("\nTest:")
print(test.isnull().sum()[test.isnull().sum() > 0])

print("\n" + "=" * 50)
print("Duplicate Rows")
print("=" * 50)

print(f"Train duplicates: {train.duplicated().sum()}")
print(f"Test duplicates : {test.duplicated().sum()}")

if 'target' in train.columns:
    print("\n" + "=" * 50)
    print("Target Statistics")
    print("=" * 50)

    print(train['target'].describe())

print("\nFirst 3 rows:")
display(train.head(3))

Dataset Shapes
Train shape : (6171, 3)
Test shape  : (4115, 3)

Target Type Distribution

Train:
target_type
tg     4143
egc    2028
Name: count, dtype: int64

Test:
target_type
tg     2763
egc    1352
Name: count, dtype: int64

Missing Values

Train:
Series([], dtype: int64)

Test:
Series([], dtype: int64)

Duplicate Rows
Train duplicates: 0
Test duplicates : 0

Target Statistics
count    6171.000000
mean       95.546584
std       109.949329
min      -118.000000
25%         4.923950
50%        57.000000
75%       179.000000
max       490.000000
Name: target, dtype: float64

First 3 rows:


,smiles,target,target_type
0,*Oc1ccc(cc1)C1(c2cc(ccc2c2ccc(cc12)[N+](=O)[O-...,294.0000,tg
1,*CCCCOC(=O)NC1CCC(CC2CCC(NC(=O)O*)CC2)CC1,6.1232,egc
2,*N1C(=O)c2c(C1=O)cc(cc2)Oc1cc2c(cc1Oc1cc3c(C(=...,222.5000,tg


In [36]:
# Split datasets by target type

train_tg = (
    train[train['target_type'] == 'tg']
    .reset_index(drop=True)
)

train_egc = (
    train[train['target_type'] == 'egc']
    .reset_index(drop=True)
)

test_tg = (
    test[test['target_type'] == 'tg']
    .reset_index(drop=True)
)

test_egc = (
    test[test['target_type'] == 'egc']
    .reset_index(drop=True)
)

print("=" * 50)
print("Dataset Split Summary")
print("=" * 50)

print(
    f"Train Tg  : {len(train_tg):,}"
)
print(
    f"Train Egc : {len(train_egc):,}"
)

print(
    f"Test Tg   : {len(test_tg):,}"
)
print(
    f"Test Egc  : {len(test_egc):,}"
)

print("\n" + "=" * 50)
print("Target Ranges")
print("=" * 50)

print(
    f"Tg range  : "
    f"[{train_tg['target'].min():.3f}, "
    f"{train_tg['target'].max():.3f}]"
)

print(
    f"Egc range : "
    f"[{train_egc['target'].min():.3f}, "
    f"{train_egc['target'].max():.3f}]"
)

print("\n" + "=" * 50)
print("Target Statistics")
print("=" * 50)

print("\nTg:")
display(train_tg['target'].describe())

print("\nEgc:")
display(train_egc['target'].describe())

Dataset Split Summary
Train Tg  : 4,143
Train Egc : 2,028
Test Tg   : 2,763
Test Egc  : 1,352

Target Ranges
Tg range  : [-118.000, 490.000]
Egc range : [0.103, 9.863]

Target Statistics

Tg:


count    4143.000000
mean      140.098547
std       109.386269
min      -118.000000
25%        55.285000
50%       132.000000
75%       230.000000
max       490.000000
Name: target, dtype: float64


Egc:


count    2028.000000
mean        4.531405
std         1.556919
min         0.103200
25%         3.286275
50%         4.613300
75%         5.810575
max         9.862700
Name: target, dtype: float64

## 2. Feature Engineering

Three complementary feature sets:
- **RDKit 2D descriptors** (~210 features): physicochemical properties (MW, logP, TPSA, ring counts, etc.)
- **Morgan fingerprints** (radius=2, 2048 bits): circular substructure encoding, great for capturing local chemistry
- **MACCS keys** (166 bits): standardised structural keys used widely in cheminformatics

In [37]:
from rdkit import Chem, DataStructs
from rdkit.Chem import (
    Descriptors,
    MACCSkeys,
    rdFingerprintGenerator
)

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold


def smiles_to_mol(smiles):
    """Convert SMILES → RDKit molecule"""

    try:
        return Chem.MolFromSmiles(smiles)
    except:
        return None


########################################################
# RDKit descriptors
########################################################

def get_rdkit_descriptors(smiles_list):

    desc_names = [
        x[0]
        for x in Descriptors.descList
    ]

    rows = []

    for smi in smiles_list:

        mol = smiles_to_mol(smi)

        if mol is None:
            rows.append(
                [np.nan] * len(desc_names)
            )
            continue

        try:

            vals = Descriptors.CalcMolDescriptors(
                mol
            )

            rows.append(
                list(vals.values())
            )

        except:

            rows.append(
                [np.nan] * len(desc_names)
            )

    return pd.DataFrame(
        rows,
        columns=desc_names
    )


########################################################
# Morgan fingerprints
########################################################

def get_morgan_fingerprints(
    smiles_list,
    radius=2,
    n_bits=2048
):

    generator = (
        rdFingerprintGenerator.GetMorganGenerator(
            radius=radius,
            fpSize=n_bits
        )
    )

    fps = np.zeros(
        (len(smiles_list), n_bits),
        dtype=np.uint8
    )

    for i, smi in enumerate(smiles_list):

        mol = smiles_to_mol(smi)

        if mol is None:
            continue

        try:

            fp = generator.GetFingerprint(
                mol
            )

            arr = np.zeros(
                n_bits,
                dtype=np.uint8
            )

            DataStructs.ConvertToNumpyArray(
                fp,
                arr
            )

            fps[i] = arr

        except:
            pass

    cols = [
        f"morgan_{i}"
        for i in range(n_bits)
    ]

    return pd.DataFrame(
        fps,
        columns=cols
    )


########################################################
# MACCS keys
########################################################

def get_maccs_keys(smiles_list):

    n_bits = 167

    fps = np.zeros(
        (len(smiles_list), n_bits),
        dtype=np.uint8
    )

    for i, smi in enumerate(smiles_list):

        mol = smiles_to_mol(smi)

        if mol is None:
            continue

        try:

            fp = MACCSkeys.GenMACCSKeys(
                mol
            )

            arr = np.zeros(
                n_bits,
                dtype=np.uint8
            )

            DataStructs.ConvertToNumpyArray(
                fp,
                arr
            )

            fps[i] = arr

        except:
            pass

    cols = [
        f"maccs_{i}"
        for i in range(n_bits)
    ]

    return pd.DataFrame(
        fps,
        columns=cols
    )


########################################################
# Main feature builder
########################################################

def build_features(
    df,
    fit_objects=None
):

    smiles = df["smiles"].tolist()

    print("Computing RDKit descriptors...")
    rdkit_df = get_rdkit_descriptors(
        smiles
    )

    print("Computing Morgan fingerprints...")
    morgan_df = get_morgan_fingerprints(
        smiles
    )

    print("Computing MACCS keys...")
    maccs_df = get_maccs_keys(
        smiles
    )

    X = pd.concat(
        [
            rdkit_df,
            morgan_df,
            maccs_df
        ],
        axis=1
    )

    ###############################################
    # Data cleanup
    ###############################################

    X = X.astype(np.float32)

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    MAX_VAL = 1e6

    X = X.mask(
        np.abs(X) > MAX_VAL,
        np.nan
    )

    ################################################
    # TRAIN
    ################################################

    if fit_objects is None:

        threshold = int(
            0.2 * len(X)
        )

        X = X.dropna(
            axis=1,
            thresh=threshold
        )

        feature_cols = (
            X.columns.tolist()
        )

        imputer = SimpleImputer(
            strategy="median"
        )

        X_imp = imputer.fit_transform(
            X
        )

        X_imp = np.nan_to_num(
            X_imp,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        X_imp = np.clip(
            X_imp,
            -1e6,
            1e6
        )

        selector = VarianceThreshold(
            threshold=0
        )

        X_final = selector.fit_transform(
            X_imp
        )

        print(
            f"Final features: {X_final.shape[1]}"
        )

        fitted = (
            imputer,
            selector,
            feature_cols
        )

        return X_final, fitted

    ################################################
    # TEST
    ################################################

    else:

        (
            imputer,
            selector,
            feature_cols
        ) = fit_objects

        X = X.reindex(
            columns=feature_cols,
            fill_value=np.nan
        )

        X_imp = imputer.transform(
            X
        )

        X_imp = np.nan_to_num(
            X_imp,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        X_imp = np.clip(
            X_imp,
            -1e6,
            1e6
        )

        X_final = selector.transform(
            X_imp
        )

        return X_final, fit_objects


print("Feature functions defined.")

Feature functions defined.


In [38]:
print("=" * 60)
print("Building Tg features (train)")
print("=" * 60)

X_tg_train, tg_feature_objects = build_features(
    train_tg
)

y_tg = train_tg['target'].values


print("\n" + "=" * 60)
print("Building Egc features (train)")
print("=" * 60)

X_egc_train, egc_feature_objects = build_features(
    train_egc
)

y_egc = train_egc['target'].values


print("\n" + "=" * 60)
print("Feature Matrix Summary")
print("=" * 60)

print(
    f"Tg X shape  : {X_tg_train.shape}"
)

print(
    f"Egc X shape : {X_egc_train.shape}"
)

print(
    f"Tg target shape  : {y_tg.shape}"
)

print(
    f"Egc target shape : {y_egc.shape}"
)

print("\nDone.")

Building Tg features (train)
Computing RDKit descriptors...
Computing Morgan fingerprints...
Computing MACCS keys...
Final features: 2347

Building Egc features (train)
Computing RDKit descriptors...
Computing Morgan fingerprints...
Computing MACCS keys...
Final features: 2302

Feature Matrix Summary
Tg X shape  : (4143, 2347)
Egc X shape : (2028, 2302)
Tg target shape  : (4143,)
Egc target shape : (2028,)

Done.


## 3. Model Training

Strategy: **5-fold CV ensemble** of LightGBM + XGBoost, averaged predictions.

LightGBM and XGBoost are complementary — they differ in their tree-building algorithms (leaf-wise vs depth-wise), so their errors partially cancel when averaged.

In [39]:
import numpy as np
import lightgbm as lgb
import xgboost as xgb

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score


def get_lgbm_params(target_type):
    """LightGBM parameters"""

    params = dict(
        objective='regression',
        metric='rmse',

        n_estimators=5000,
        learning_rate=0.01,

        num_leaves=31,
        max_depth=-1,
        min_child_samples=30,

        subsample=0.8,
        colsample_bytree=0.7,

        reg_alpha=0.5,
        reg_lambda=2.0,

        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )

    if target_type == 'tg':
        params['num_leaves'] = 63
        params['colsample_bytree'] = 0.8

    return params


def get_xgb_params(target_type):
    """XGBoost parameters"""

    params = dict(
        objective='reg:squarederror',

        n_estimators=5000,
        learning_rate=0.01,

        max_depth=5,
        min_child_weight=10,

        subsample=0.8,
        colsample_bytree=0.7,

        gamma=0.1,
        reg_alpha=0.5,
        reg_lambda=2.0,

        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',

        early_stopping_rounds=100
    )

    if target_type == 'egc':
        params['max_depth'] = 4

    return params


def train_cv_ensemble(
    X_train,
    y_train,
    X_test,
    target_type,
    n_splits=5
):

    print("="*60)
    print(f"Training {target_type.upper()} model")
    print("="*60)

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SEED
    )

    oof_lgbm = np.zeros(len(X_train))
    oof_xgb = np.zeros(len(X_train))

    test_lgbm = np.zeros(len(X_test))
    test_xgb = np.zeros(len(X_test))

    fold_r2_lgbm = []
    fold_r2_xgb = []

    lgbm_params = get_lgbm_params(target_type)
    xgb_params = get_xgb_params(target_type)

    feature_importance = np.zeros(X_train.shape[1])

    for fold, (tr_idx, val_idx) in enumerate(
        kf.split(X_train),
        start=1
    ):

        print(f"\nFold {fold}/{n_splits}")

        X_tr = X_train[tr_idx]
        X_val = X_train[val_idx]

        y_tr = y_train[tr_idx]
        y_val = y_train[val_idx]

        ##################################
        # LIGHTGBM
        ##################################

        model_lgbm = lgb.LGBMRegressor(
            **lgbm_params
        )

        model_lgbm.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(
                    100,
                    verbose=False
                )
            ]
        )

        best_iter_lgbm = (
            model_lgbm.best_iteration_
            if model_lgbm.best_iteration_
            else lgbm_params['n_estimators']
        )

        pred_val_lgbm = model_lgbm.predict(
            X_val,
            num_iteration=best_iter_lgbm
        )

        pred_test_lgbm = model_lgbm.predict(
            X_test,
            num_iteration=best_iter_lgbm
        )

        oof_lgbm[val_idx] = pred_val_lgbm

        test_lgbm += (
            pred_test_lgbm / n_splits
        )

        r2_l = r2_score(
            y_val,
            pred_val_lgbm
        )

        fold_r2_lgbm.append(r2_l)

        feature_importance += (
            model_lgbm.feature_importances_
            / n_splits
        )

        ##################################
        # XGBOOST
        ##################################

        model_xgb = xgb.XGBRegressor(
            **xgb_params
        )

        model_xgb.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        best_iter_xgb = (
            model_xgb.best_iteration
            if model_xgb.best_iteration
            else xgb_params['n_estimators']
        )

        pred_val_xgb = model_xgb.predict(
            X_val,
            iteration_range=(
                0,
                best_iter_xgb + 1
            )
        )

        pred_test_xgb = model_xgb.predict(
            X_test,
            iteration_range=(
                0,
                best_iter_xgb + 1
            )
        )

        oof_xgb[val_idx] = pred_val_xgb

        test_xgb += (
            pred_test_xgb / n_splits
        )

        r2_x = r2_score(
            y_val,
            pred_val_xgb
        )

        fold_r2_xgb.append(r2_x)

        print(
            f"LGBM R²={r2_l:.5f} | "
            f"XGB R²={r2_x:.5f}"
        )

    ##################################
    # ENSEMBLE
    ##################################

    w_lgbm = max(
        np.mean(fold_r2_lgbm),
        0
    )

    w_xgb = max(
        np.mean(fold_r2_xgb),
        0
    )

    total = w_lgbm + w_xgb

    if total == 0:
        w_lgbm = 0.5
        w_xgb = 0.5
    else:
        w_lgbm /= total
        w_xgb /= total

    print("\nEnsemble weights")
    print(
        f"LGBM={w_lgbm:.3f}"
    )
    print(
        f"XGB={w_xgb:.3f}"
    )

    oof_ensemble = (
        w_lgbm * oof_lgbm +
        w_xgb * oof_xgb
    )

    test_ensemble = (
        w_lgbm * test_lgbm +
        w_xgb * test_xgb
    )

    oof_r2 = r2_score(
        y_train,
        oof_ensemble
    )

    print("\nFinal Results")
    print(
        f"OOF Ensemble R²: {oof_r2:.5f}"
    )

    print(
        f"LGBM Mean: "
        f"{np.mean(fold_r2_lgbm):.5f}"
    )

    print(
        f"XGB Mean: "
        f"{np.mean(fold_r2_xgb):.5f}"
    )

    return (
        oof_ensemble,
        test_ensemble,
        oof_r2
    )


print("Training functions defined.")

Training functions defined.


In [40]:
print("=" * 60)
print("Building Tg features (test)")
print("=" * 60)

X_tg_test, _ = build_features(
    test_tg,
    fit_objects=tg_feature_objects
)


print("\n" + "=" * 60)
print("Building Egc features (test)")
print("=" * 60)

X_egc_test, _ = build_features(
    test_egc,
    fit_objects=egc_feature_objects
)


print("\n" + "=" * 60)
print("Feature Matrix Summary")
print("=" * 60)

print(
    f"Tg test shape  : {X_tg_test.shape}"
)

print(
    f"Egc test shape : {X_egc_test.shape}"
)

print("\nDone.")

Building Tg features (test)
Computing RDKit descriptors...
Computing Morgan fingerprints...
Computing MACCS keys...

Building Egc features (test)
Computing RDKit descriptors...
Computing Morgan fingerprints...
Computing MACCS keys...

Feature Matrix Summary
Tg test shape  : (2763, 2347)
Egc test shape : (1352, 2302)

Done.


In [41]:
print("=" * 60)
print("TRAINING Tg MODELS")
print("=" * 60)

oof_tg, pred_tg, r2_tg = train_cv_ensemble(
    X_train=X_tg_train,
    y_train=y_tg,
    X_test=X_tg_test,
    target_type='tg'
)


print("\n" + "=" * 60)
print("TRAINING Egc MODELS")
print("=" * 60)

oof_egc, pred_egc, r2_egc = train_cv_ensemble(
    X_train=X_egc_train,
    y_train=y_egc,
    X_test=X_egc_test,
    target_type='egc'
)


#################################################
# Summary
#################################################

print("\n" + "=" * 60)
print("FINAL CV SUMMARY")
print("=" * 60)

mean_r2 = np.nanmean(
    [r2_tg, r2_egc]
)

print(
    f"Tg OOF R²   : {r2_tg:.5f}"
)

print(
    f"Egc OOF R²  : {r2_egc:.5f}"
)

print(
    f"Mean OOF R² : {mean_r2:.5f}"
)

print("=" * 60)
print(
    "Competition metric proxy = "
    "mean(Tg_R², Egc_R²)"
)
print("=" * 60)

TRAINING Tg MODELS
Training TG model

Fold 1/5
LGBM R²=0.88414 | XGB R²=0.88680

Fold 2/5
LGBM R²=0.90169 | XGB R²=0.90681

Fold 3/5
LGBM R²=0.88642 | XGB R²=0.89514

Fold 4/5
LGBM R²=0.89278 | XGB R²=0.89643

Fold 5/5
LGBM R²=0.87372 | XGB R²=0.87849

Ensemble weights
LGBM=0.499
XGB=0.501

Final Results
OOF Ensemble R²: 0.89240
LGBM Mean: 0.88775
XGB Mean: 0.89273

TRAINING Egc MODELS
Training EGC model

Fold 1/5
LGBM R²=0.86816 | XGB R²=0.87925

Fold 2/5
LGBM R²=0.89464 | XGB R²=0.89843

Fold 3/5
LGBM R²=0.90738 | XGB R²=0.91008

Fold 4/5
LGBM R²=0.90749 | XGB R²=0.90521

Fold 5/5
LGBM R²=0.91679 | XGB R²=0.92347

Ensemble weights
LGBM=0.499
XGB=0.501

Final Results
OOF Ensemble R²: 0.90353
LGBM Mean: 0.89889
XGB Mean: 0.90329

FINAL CV SUMMARY
Tg OOF R²   : 0.89240
Egc OOF R²  : 0.90353
Mean OOF R² : 0.89796
Competition metric proxy = mean(Tg_R², Egc_R²)


## 4. Generate Submission

In [42]:
#################################################
# Build submission
#################################################

test_tg_out = test_tg[['id']].copy()
test_tg_out['target'] = pred_tg

test_egc_out = test_egc[['id']].copy()
test_egc_out['target'] = pred_egc


submission = pd.concat(
    [
        test_tg_out,
        test_egc_out
    ],
    axis=0
)

submission = (
    submission
    .sort_values('id')
    .reset_index(drop=True)
)

#################################################
# Safety checks
#################################################

assert len(submission) == len(test), \
    "Submission row count mismatch"

assert submission['target'].isna().sum() == 0, \
    "NaN predictions found"

assert np.isfinite(
    submission['target']
).all(), \
    "Infinite values found"

#################################################
# Diagnostics
#################################################

print("="*60)
print("Submission Summary")
print("="*60)

print(
    f"Submission shape: "
    f"{submission.shape}"
)

print(
    "\nPrediction statistics:"
)

display(
    submission['target']
    .describe()
)

print("\nFirst 10 rows:")

display(
    submission.head(10)
)

#################################################
# Save
#################################################

submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

np.save("/kaggle/working/pred_tg.npy", pred_tg)
np.save("/kaggle/working/pred_egc.npy", pred_egc)
print(f"\nSubmission saved: {submission_path}")

Submission Summary
Submission shape: (4115, 2)

Prediction statistics:


count    4115.000000
mean       96.837220
std       105.867634
min      -102.820700
25%         5.009453
50%        61.936847
75%       179.597104
max       404.777590
Name: target, dtype: float64


First 10 rows:


,id,target
0,1,303.052343
1,2,4.990908
2,3,64.375578
3,4,44.940457
4,5,87.754154
5,6,139.565934
6,7,85.284205
7,8,229.983003
8,9,250.605709
9,10,6.293798



Submission saved: /kaggle/working/submission.csv


## 5. Quick Sanity Checks

In [43]:
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

#################################################
# Tg plot
#################################################

plt.figure(figsize=(6,5))

plt.scatter(
    y_tg,
    oof_tg,
    alpha=0.3,
    s=10
)

mn = min(
    y_tg.min(),
    oof_tg.min()
)

mx = max(
    y_tg.max(),
    oof_tg.max()
)

plt.plot(
    [mn, mx],
    [mn, mx],
    '--',
    linewidth=1
)

plt.xlabel('True Tg (°C)')
plt.ylabel('Predicted Tg (°C)')
plt.title(
    f'Tg OOF | R² = {r2_tg:.4f}'
)

plt.tight_layout()

plt.savefig(
    '/kaggle/working/tg_oof_scatter.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()


#################################################
# Egc plot
#################################################

plt.figure(figsize=(6,5))

plt.scatter(
    y_egc,
    oof_egc,
    alpha=0.3,
    s=10
)

mn = min(
    y_egc.min(),
    oof_egc.min()
)

mx = max(
    y_egc.max(),
    oof_egc.max()
)

plt.plot(
    [mn, mx],
    [mn, mx],
    '--',
    linewidth=1
)

plt.xlabel('True Egc (eV)')
plt.ylabel('Predicted Egc (eV)')
plt.title(
    f'Egc OOF | R² = {r2_egc:.4f}'
)

plt.tight_layout()

plt.savefig(
    '/kaggle/working/egc_oof_scatter.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("Saved:")
print("/kaggle/working/tg_oof_scatter.png")
print("/kaggle/working/egc_oof_scatter.png")

Saved:
/kaggle/working/tg_oof_scatter.png
/kaggle/working/egc_oof_scatter.png


In [44]:
#################################################
# Prediction sanity checks
#################################################

print("=" * 60)
print("PREDICTION SANITY CHECK")
print("=" * 60)

# Tg
print("\nTg predictions")
print("-" * 40)

print(
    f"Min   : {pred_tg.min():.2f}"
)
print(
    f"Max   : {pred_tg.max():.2f}"
)
print(
    f"Mean  : {pred_tg.mean():.2f}"
)
print(
    f"Std   : {pred_tg.std():.2f}"
)

print(
    f"Train range : "
    f"[{y_tg.min():.2f}, {y_tg.max():.2f}]"
)

tg_outside = (
    (pred_tg < y_tg.min()) |
    (pred_tg > y_tg.max())
).sum()

print(
    f"Outside train range: "
    f"{tg_outside}/{len(pred_tg)}"
)


# Egc
print("\nEgc predictions")
print("-" * 40)

print(
    f"Min   : {pred_egc.min():.4f}"
)
print(
    f"Max   : {pred_egc.max():.4f}"
)
print(
    f"Mean  : {pred_egc.mean():.4f}"
)
print(
    f"Std   : {pred_egc.std():.4f}"
)

print(
    f"Train range : "
    f"[{y_egc.min():.4f}, {y_egc.max():.4f}]"
)

egc_outside = (
    (pred_egc < y_egc.min()) |
    (pred_egc > y_egc.max())
).sum()

print(
    f"Outside train range: "
    f"{egc_outside}/{len(pred_egc)}"
)


#################################################
# Submission checks
#################################################

print("\n" + "=" * 60)
print("SUBMISSION VALIDATION")
print("=" * 60)

nan_count = submission['target'].isna().sum()

inf_count = (
    ~np.isfinite(
        submission['target']
    )
).sum()

print(
    f"NaNs      : {nan_count}"
)

print(
    f"Infinities: {inf_count}"
)

print(
    f"Rows       : {len(submission)}"
)

if nan_count == 0 and inf_count == 0:
    print("\nAll checks passed")
else:
    print("\nWarning: invalid predictions detected")

PREDICTION SANITY CHECK

Tg predictions
----------------------------------------
Min   : -102.82
Max   : 404.78
Mean  : 142.01
Std   : 102.36
Train range : [-118.00, 490.00]
Outside train range: 0/2763

Egc predictions
----------------------------------------
Min   : 0.6872
Max   : 8.2373
Mean  : 4.5247
Std   : 1.4428
Train range : [0.1032, 9.8627]
Outside train range: 0/1352

SUBMISSION VALIDATION
NaNs      : 0
Infinities: 0
Rows       : 4115

All checks passed
